In [1]:
import pandas as pd 
import numpy as np

In [4]:
pd.read_csv('output/monthlystats_aqi_by_county_2016-2025.csv')#.keys()

,Unnamed: 0,State Name,county Name,Year,Month,State Code,County Code,Avg AQI,Category,Min AQI,...,DP as NO2 (AQI>50),DP as Ozone (AQI>50),DP as PM10 (AQI>50),DP as PM2.5 (AQI>50),Days with AQI >100,Monthly Mode DP (AQI>100),DP as NO2 (AQI>100),DP as Ozone (AQI>100),DP as PM10 (AQI>100),DP as PM2.5 (AQI>100)
0,0,Illinois,Adams,2016,4,17,1,37.200000,Good,22,...,0.0,2.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
1,1,Illinois,Adams,2016,5,17,1,42.419355,Good,21,...,0.0,3.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
2,2,Illinois,Adams,2016,6,17,1,49.633333,Good,35,...,0.0,8.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
3,3,Illinois,Adams,2016,7,17,1,34.774194,Good,29,...,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
4,4,Illinois,Adams,2016,8,17,1,32.548387,Good,23,...,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2487,2487,Illinois,Winnebago,2025,2,17,201,35.240000,Good,18,...,0.0,0.0,0.0,3.0,0.0,0,0.0,0.0,0.0,0.0
2488,2488,Illinois,Winnebago,2025,3,17,201,38.064516,Good,15,...,0.0,0.0,0.0,3.0,0.0,0,0.0,0.0,0.0,0.0
2489,2489,Illinois,Winnebago,2025,4,17,201,43.066667,Good,31,...,0.0,2.0,0.0,5.0,0.0,0,0.0,0.0,0.0,0.0
2490,2490,Illinois,Winnebago,2025,5,17,201,42.064516,Good,24,...,0.0,6.0,0.0,1.0,0.0,0,0.0,0.0,0.0,0.0


In [117]:
# monthly - monthly mean AQI for each year at each county 
# yearly - climatological mean AQI for each year 
# 

In [6]:
df = pd.read_csv('daily_aqi_by_county_2016-2025.csv')
df['Date'] = pd.to_datetime(df['Date'])
# Not all counties are covered 
# Future work: add confidence in AQI levels by number of sites reporting - weighted by distance? 
### Counties w/o data: use stations nearby them to interpolate?? 

#### Monthly

In [9]:
# Get Monthly Avg, Max, Min 
monthly = df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['AQI'].mean().reset_index().rename(columns={'AQI': 'Avg AQI'})
monthly['Min AQI'] = df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['AQI'].min().reset_index()['AQI']
monthly['Max AQI'] = df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['AQI'].max().reset_index()['AQI']
maxdefparam_idx = df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['AQI'].idxmax()
mindefparam_idx = df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['AQI'].idxmin()
# monthly['Max Defining Parameter'] = df.loc[maxdefparam_idx, ['Defining Parameter']].reset_index(drop=True) #of the aqi max day 
# monthly['Min Defining Parameter'] = df.loc[maxdefparam_idx, ['Defining Parameter']].reset_index(drop=True) # of the aqi min day 

In [10]:
# days above AQI threshold and the Mode Defining Parameter (DP)
count = (df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')]))['Category'].value_counts().reset_index()
dummies = pd.get_dummies(count['Category']) #check dates 
result = dummies.mul(count['count'], axis=0)
mod_or_worse = count.drop(count[count['Category'] == 'Good'].index).groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['count'].sum().reset_index().rename(columns={'count': 'Days with AQI >50'})
unhealthy_or_worse = count.drop(count[(count['Category'].isin(['Good','Moderate']))].index).groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['count'].sum().reset_index().rename(columns={'count': 'Days with AQI >100'})

count_dp_aqi50 = df.drop(df[df['Category'] == 'Good'].index).groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['Defining Parameter'].value_counts().reset_index()
maxdefparam_idx = count_dp_aqi50.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['count'].idxmax()
mod_or_worse['Monthly Mode DP (AQI>50)']= count_dp_aqi50.loc[maxdefparam_idx, ['Defining Parameter']].reset_index()['Defining Parameter']


count_dp_aqi100 = df.drop(df[(df['Category'].isin(['Good','Moderate']))].index).groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['Defining Parameter'].value_counts().reset_index()
maxdefparam_idx = count_dp_aqi100.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['count'].idxmax()
unhealthy_or_worse['Monthly Mode DP (AQI>100)'] = count_dp_aqi100.loc[maxdefparam_idx, ['Defining Parameter']].reset_index()['Defining Parameter']

no_good = pd.merge(mod_or_worse, unhealthy_or_worse, on=['county Name','Date'], how='outer').fillna(0)
    # rename(columns={'count_x': 'Days with AQI >50','count_y':'Days with AQI >100'}, inplace=False)

monthly = pd.merge(monthly,no_good, on=['county Name','Date'], how='outer').fillna(0)

In [11]:
# DP on days above above threshold count 
count_dp_aqi50 = df.drop(df[df['Category'] == 'Good'].index).groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['Defining Parameter'].value_counts().unstack().reset_index()
count_dp_aqi100 = df.drop(df[(df['Category'].isin(['Good','Moderate']))].index).groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['Defining Parameter'].value_counts().unstack().reset_index()
result_dp = pd.merge(count_dp_aqi50,count_dp_aqi100, on=['county Name','Date'], how='outer').fillna(0).rename(columns={'NO2_x':'DP as NO2 (AQI>50)','Ozone_x':'DP as Ozone (AQI>50)','PM10_x':'DP as PM10 (AQI>50)','PM2.5_x':'DP as PM2.5 (AQI>50)',\
                                             'NO2_y':'DP as NO2 (AQI>100)','Ozone_y':'DP as Ozone (AQI>100)','PM10_y':'DP as PM10 (AQI>100)','PM2.5_y':'DP as PM2.5 (AQI>100)'},inplace=False)

mutliidx = monthly.loc[:,['county Name','Date']]# contains each months of all years 
dp = pd.merge(mutliidx,result_dp, on=['county Name','Date'], how='outer').fillna(0) #DP for each month with county/Date 

In [122]:
# Mode Category - will mainly be good - so keeping avg category 
# count_cat = df.groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['Category'].value_counts()
# maxdefparam_idx = count_dp_aqi50.reset_index().groupby(['county Name', pd.Grouper(key='Date', freq='ME')])['count'].idxmax()
# mode_dp_aqi50 = count_dp_aqi50.reset_index().loc[maxdefparam_idx, ['Defining Parameter']].reset_index()['Defining Parameter']
# result_dp['Monthly Mode DP (AQI>50)'] = mode_dp_aqi50

In [12]:
### Mark Category (good or moderate)
conditions = [
    monthly['Avg AQI'] <= 50,
    (monthly['Avg AQI'] > 50) & (monthly['Avg AQI'] <= 100),
    (monthly['Avg AQI'] > 100) & (monthly['Avg AQI'] <= 150),
    (monthly['Avg AQI'] > 150) & (monthly['Avg AQI'] <= 200),
    (monthly['Avg AQI'] > 200) & (monthly['Avg AQI'] <= 300),
    (monthly['Avg AQI'] > 300),
]
choices = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups','Unhealthy','Very Unhealthy','Hazardous']
monthly['Category'] = np.select(conditions, choices, default='Other')
# monthly

In [13]:
# Compile all monthly stats 
monthlystats = pd.concat([monthly, dp.drop(['county Name', 'Date'], axis=1)], axis=1)

###  Drop dates for monthly 
monthlystats['Year'] = monthlystats['Date'].dt.year
monthlystats['Month'] = monthlystats['Date'].dt.month
monthlystats = monthlystats.drop('Date',axis=1)
monthlystats.keys()

###  Add FIPS 
county_code = df.groupby(['State Name','county Name'])['County Code'].unique().reset_index()
state_code = df.groupby(['State Name','county Name'])['State Code'].unique().reset_index()
state_code['State Code'] = state_code['State Code'].astype(int)
county_code['County Code'] = county_code['County Code'].astype(int)
state_code['County Code'] = county_code['County Code']

monthlystats = pd.merge(monthlystats,state_code, on=['county Name'], how='outer')

### Reorder cols 

new_order = ['State Name','county Name','Year', 'Month', 'State Code', 'County Code','Avg AQI','Category', 'Min AQI', 'Max AQI',
             'Days with AQI >50', 'Monthly Mode DP (AQI>50)','DP as NO2 (AQI>50)','DP as Ozone (AQI>50)', 'DP as PM10 (AQI>50)', 'DP as PM2.5 (AQI>50)',
             'Days with AQI >100','Monthly Mode DP (AQI>100)','DP as NO2 (AQI>100)', 'DP as Ozone (AQI>100)', 'DP as PM10 (AQI>100)','DP as PM2.5 (AQI>100)']
monthlystats = monthlystats[new_order]
# monthlystats.to_csv('output/monthlystats_aqi_by_county_2016-2025.csv')

#### Yearly

In [15]:
yearlystats = monthlystats.groupby(['State Name','county Name','Month'])['Avg AQI'].mean().reset_index()#.rename(columns={'AQI': 'Avg AQI'})
yearlystats['Avg Min AQI'] = monthlystats.groupby(['State Name','county Name','Month'])['Min AQI'].mean().reset_index()['Min AQI']
yearlystats['Avg Max AQI'] = monthlystats.groupby(['State Name','county Name','Month'])['Max AQI'].mean().reset_index()['Max AQI']
yearlystats['Avg Days AQI >50'] = monthlystats.groupby(['State Name','county Name','Month'])['Days with AQI >50'].mean().reset_index()['Days with AQI >50']#.rename(columns={'AQI': 'Avg AQI'})
yearlystats['Avg Days AQI >100'] = monthlystats.groupby(['State Name','county Name','Month'])['Days with AQI >100'].mean().reset_index()['Days with AQI >100']
yearlystats['Total Days AQI >50'] = monthlystats.groupby(['State Name','county Name','Month'])['Days with AQI >50'].sum().reset_index()['Days with AQI >50']
yearlystats['Total Days AQI >100'] = monthlystats.groupby(['State Name','county Name','Month'])['Days with AQI >100'].sum().reset_index()['Days with AQI >100']

# monthly: (took average of across days for each month) per month per year per county 
# yearly: (took average of each month across years) per county per month | per county per year?? 
# summary: (entire year) per county 

In [16]:
count_dp_aqi100 = monthlystats.drop(monthlystats[monthlystats['Monthly Mode DP (AQI>100)'] == 0].index).groupby(['State Name','county Name','Month'])['Monthly Mode DP (AQI>100)'].value_counts().reset_index()
maxdefparam_idx = count_dp_aqi100.groupby(['State Name','county Name','Month'])['count'].idxmax()
dpmode_100 = count_dp_aqi100.loc[maxdefparam_idx, ['State Name','county Name','Month','Monthly Mode DP (AQI>100)']]

count_dp_aqi50 = monthlystats.drop(monthlystats[monthlystats['Monthly Mode DP (AQI>50)'] == 0].index).groupby(['State Name','county Name','Month'])['Monthly Mode DP (AQI>50)'].value_counts().reset_index()
maxdefparam_idx = count_dp_aqi50.groupby(['State Name','county Name','Month'])['count'].idxmax()
dpmode_50 = count_dp_aqi50.loc[maxdefparam_idx, ['State Name','county Name','Month','Monthly Mode DP (AQI>50)']]

dpmode = pd.merge(dpmode_50,dpmode_100,how='outer')

idx = yearlystats.loc[:,['State Name','county Name','Month']]
dpmode = pd.merge(idx,dpmode,how='outer')

yearlystats['Monthly Mode DP (AQI>50)'] =dpmode['Monthly Mode DP (AQI>50)']
yearlystats['Monthly Mode DP (AQI>100)'] =dpmode['Monthly Mode DP (AQI>100)']

In [17]:
### Add FIPS 
yearlystats = pd.merge(yearlystats,state_code, on=['State Name','county Name'], how='outer')
yearlystats.keys()

Index(['State Name', 'county Name', 'Month', 'Avg AQI', 'Avg Min AQI',
       'Avg Max AQI', 'Avg Days AQI >50', 'Avg Days AQI >100',
       'Total Days AQI >50', 'Total Days AQI >100', 'Monthly Mode DP (AQI>50)',
       'Monthly Mode DP (AQI>100)', 'State Code', 'County Code'],
      dtype='object')

In [18]:
### Reorder cols 
new_order = ['State Name', 'county Name', 'State Code', 'County Code', 'Month', 'Avg AQI', 'Avg Min AQI','Avg Max AQI',
             'Avg Days AQI >50', 'Avg Days AQI >100', 'Total Days AQI >50', 'Total Days AQI >100', 'Monthly Mode DP (AQI>50)',
       'Monthly Mode DP (AQI>100)']
yearlystats = yearlystats[new_order]
# yearlystats.to_csv('output/yearlystats_aqi_by_county_2016-2025.csv')

In [19]:
yearlystats

,State Name,county Name,State Code,County Code,Month,Avg AQI,Avg Min AQI,Avg Max AQI,Avg Days AQI >50,Avg Days AQI >100,Total Days AQI >50,Total Days AQI >100,Monthly Mode DP (AQI>50),Monthly Mode DP (AQI>100)
0,Illinois,Adams,17,1,3,38.655197,25.000000,52.444444,0.777778,0.000000,7.0,0.0,Ozone,NaN
1,Illinois,Adams,17,1,4,42.436450,26.400000,65.900000,3.200000,0.000000,32.0,0.0,Ozone,NaN
2,Illinois,Adams,17,1,5,45.402452,27.700000,72.500000,5.700000,0.300000,57.0,3.0,Ozone,Ozone
3,Illinois,Adams,17,1,6,50.697328,29.800000,92.000000,8.500000,1.000000,85.0,10.0,Ozone,Ozone
4,Illinois,Adams,17,1,7,40.557300,28.333333,57.111111,2.222222,0.000000,20.0,0.0,Ozone,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,Illinois,Winnebago,17,201,8,45.405018,25.555556,67.777778,10.222222,0.000000,92.0,0.0,PM2.5,NaN
264,Illinois,Winnebago,17,201,9,41.899177,19.111111,73.777778,8.555556,0.333333,77.0,3.0,PM2.5,Ozone
265,Illinois,Winnebago,17,201,10,38.351254,19.777778,62.111111,7.555556,0.000000,68.0,0.0,PM2.5,NaN
266,Illinois,Winnebago,17,201,11,39.160157,15.000000,68.555556,8.888889,0.000000,80.0,0.0,PM2.5,NaN


#### Summary

In [21]:
# 'Avg Days AQI >50', 'Avg Days AQI >100', 'Total Days AQI >50', 'Total Days AQI >100',
summary = state_code.loc[:,['State Name', 'county Name', 'State Code', 'County Code']]
summary['Annual AQI'] = monthlystats.groupby(['State Name','county Name'])['Avg AQI'].mean().reset_index()['Avg AQI']
summary['Days with AQI >50 Each Year'] = monthlystats.groupby(['State Name','county Name'])['Days with AQI >50'].mean().reset_index()['Days with AQI >50']
summary['Days with AQI >100 Each Year'] = monthlystats.groupby(['State Name','county Name'])['Days with AQI >100'].mean().reset_index()['Days with AQI >100']
summary['Total Days with AQI >50'] = monthlystats.groupby(['State Name','county Name'])['Days with AQI >50'].sum().reset_index()['Days with AQI >50']
summary['Total Days with AQI >100'] = monthlystats.groupby(['State Name','county Name'])['Days with AQI >100'].sum().reset_index()['Days with AQI >100']

# summary.to_csv('output/summarystats_aqi_by_county_2016-2025.csv')
summary

,State Name,county Name,State Code,County Code,Annual AQI,Days with AQI >50 Each Year,Days with AQI >100 Each Year,Total Days with AQI >50,Total Days with AQI >100
0,Illinois,Adams,17,1,40.763172,3.226667,0.173333,242.0,13.0
1,Illinois,Champaign,17,19,46.461317,11.078947,0.192982,1263.0,22.0
2,Illinois,Clark,17,23,35.981609,2.087379,0.087379,215.0,9.0
3,Illinois,Cook,17,31,59.673772,21.400000,1.443478,2461.0,166.0
4,Illinois,DuPage,17,43,48.489304,12.833333,0.385965,1463.0,44.0
5,Illinois,Effingham,17,49,41.022680,3.479452,0.205479,254.0,15.0
6,Illinois,Hamilton,17,65,46.228318,11.157895,0.166667,1272.0,19.0
7,Illinois,Jersey,17,83,46.026499,10.176991,0.362832,1150.0,41.0
8,Illinois,Jo Daviess,17,85,35.775570,1.830357,0.116071,205.0,13.0
9,Illinois,Kane,17,89,44.981585,7.359649,0.482456,839.0,55.0
